# Chapter 5: Mixed Strategy Nash Equilibria, with Gambit

This notebook accompanies Chapter 5 of *An Introduction to Computational Game Theory*.
It recomputes the chapter's examples with Gambit (`pygambit` 17): mixed strategy
profiles and expected payoffs, the $2\times 2$ procedure and the difference trick, best
response correspondences, the best response condition, degeneracy, Nash's map, and an
$n$-player game whose only mixed equilibrium is irrational.

Notation as in the book: $x$ and $y$ are the mixed strategies of players 1 and 2, $A$
and $B$ their payoff matrices (rows = player 1's strategies). Exact fractions are used
throughout; Gambit computes with rationals by default.

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cgtbook import (
    P1_COLOUR,
    bimatrix,
    eq_table,
    frac,
    nice,
    plot_br_2x2,
    plot_upper_envelope,
    pure_profile_names,
    show,
    table_game,
)

import pygambit as gbt

## Matching Pennies has no pure equilibrium

The chapter opens with Matching Pennies.

In [ ]:
mp = bimatrix([[1, -1], [-1, 1]], [[-1, 1], [1, -1]],
              ["Heads", "Tails"], ["heads", "tails"], title="Matching Pennies")
print("pure equilibria:", gbt.nash.enumpure_solve(mp).equilibria)

A **mixed strategy profile** in Gambit assigns a probability to each strategy.
`payoffs` gives each player's expected utility $EU_i$, and `strategy_values` the expected
payoff of each pure strategy against the other player's mixed strategy (the quantities
$(Ay)_i$ and $(x^\top B)_j$ of the best response condition).

In [ ]:
x = [Fraction(3, 4), Fraction(1, 4)]
y = [Fraction(1, 3), Fraction(2, 3)]
profile = mp.mixed_strategy_profile([x, y], rational=True)
print(show(profile))
print("expected payoffs EU_i:     ", nice(profile.payoffs))
print("pure strategy values:      ", nice(profile.strategy_values))

Against $y = (1/3, 2/3)$ player 1 strictly prefers Tails, so $x = (3/4, 1/4)$ is not a
best response. Gambit's `enummixed_solve` (an algorithm of Chapter 6) finds all
equilibria of a two-player game: here the unique equilibrium
$x = y = (1/2, 1/2)$, with expected payoff 0 to both players.

In [ ]:
eq_table(gbt.nash.enummixed_solve(mp).equilibria)

## 5.1 Computing mixed strategy equilibria of $2\times 2$ games

The book's procedure for a non-degenerate $2\times 2$ game: find the pure equilibria;
if there is exactly one, stop; otherwise there is exactly one more equilibrium, fully
mixed, in which each player makes the *other* indifferent. The **difference trick**
computes it: the absolute payoff difference of player 1 in each column becomes the
weight of the *other* column, and similarly for player 2 with rows.

In [ ]:
def difference_trick(A, B):
    """The fully mixed equilibrium candidate (x, y) of a 2x2 game by the difference trick."""
    A = [[Fraction(a) for a in row] for row in A]
    B = [[Fraction(b) for b in row] for row in B]
    d_left, d_right = abs(A[0][0] - A[1][0]), abs(A[0][1] - A[1][1])   # player 1, by column
    d_top, d_bottom = abs(B[0][0] - B[0][1]), abs(B[1][0] - B[1][1])   # player 2, by row
    y = (d_right / (d_left + d_right), d_left / (d_left + d_right))
    x = (d_bottom / (d_top + d_bottom), d_top / (d_top + d_bottom))
    return x, y


def solve_2x2(A, B, rows=("Top", "Bottom"), cols=("left", "right")):
    g = bimatrix(A, B, list(rows), list(cols))
    pure = [pure_profile_names(eq) for eq in gbt.nash.enumpure_solve(g).equilibria]
    if len(pure) == 1:
        return pure, None
    return pure, difference_trick(A, B)


def fmt(v):
    return "(" + ", ".join(str(a) for a in v) + ")"


games_2x2 = {
    "Matching Pennies": ([[1, -1], [-1, 1]], [[-1, 1], [1, -1]]),
    "Difference-trick game": ([[0, 2], [1, 0]], [[3, 0], [0, 3]]),
    "Matching Pennies variant": ([[1, -1], [-1, 1]], [[0, 1], [1, -1]]),
    "Nash's Theorem example": ([[1, 0], [0, 2]], [[5, 0], [0, 1]]),
}
rows = []
for name, (A, B) in games_2x2.items():
    pure, mixed = solve_2x2(A, B)
    g = bimatrix(A, B, ["Top", "Bottom"], ["left", "right"])
    rows.append({
        "game": name,
        "pure NE": pure,
        "difference trick: x": fmt(mixed[0]) if mixed else "-",
        "difference trick: y": fmt(mixed[1]) if mixed else "-",
        "Gambit enummixed": [show(eq) for eq in gbt.nash.enummixed_solve(g).equilibria],
    })
pd.DataFrame(rows).set_index("game")

These agree with the book:
* Difference-trick game: $x = (1/2, 1/2)$, $y = (2/3, 1/3)$.
* Matching Pennies variant: player 1 plays Heads with probability $2/3$, while player 2
  still mixes $(1/2, 1/2)$.
* Nash's Theorem example: pure equilibria (Top, left) and (Bottom, right), and the mixed
  equilibrium with prob(Top) $= 1/6$, prob(right) $= 1/3$.

At the mixed equilibrium of the difference-trick game both of player 1's pure strategies
are worth $2/3$ and both of player 2's are worth $3/2$:

In [ ]:
eq = gbt.nash.enummixed_solve(bimatrix([[0, 2], [1, 0]], [[3, 0], [0, 3]],
                                        ["T", "B"], ["l", "r"])).equilibria[0]
nice(eq.strategy_values)

## 5.2 Nash's Theorem for $2\times 2$ games: best response correspondences

The book draws the best response correspondences in the unit square with prob(right)
on the horizontal axis and prob(Top) on the vertical axis. Equilibria are the
intersections; here they are computed by Gambit and marked as black dots.

In [ ]:
def br_picture(A, B, ax, title):
    g = bimatrix(A, B, ["Top", "Bottom"], ["left", "right"])
    res = gbt.nash.enummixed_solve(g, cliques=True)

    def point(eq):
        return (eq["2"]["right"], eq["1"]["Top"])
    pts = [point(eq) for eq in res.equilibria]
    # in a 2x2 game a maximal Nash subset has at most two extreme points: draw the segment
    segs = [(point(c[0]), point(c[1])) for c in res.cliques if len(c) == 2]
    plot_br_2x2(A, B, ax=ax, equilibria=pts, segments=segs, title=title)
    return res


fig, ax = plt.subplots(figsize=(4, 4))
br_picture([[1, 0], [0, 2]], [[5, 0], [0, 1]], ax, "Nash's Theorem example")
ax.legend(loc="upper right", fontsize=8)
plt.show()

The four examples of Figure `fig:2x2brcexamples`:

In [ ]:
panels = {
    "(a) two pure equilibria off the diagonal": ([[0, 1], [2, 0]], [[0, 5], [1, 0]]),
    "(b) unique mixed equilibrium": ([[0, 1], [2, 0]], [[5, 0], [0, 1]]),
    "(c) strict dominance": ([[0, 1], [2, 0]], [[5, 0], [1, 0]]),
    "(d) degenerate game": ([[0, 1], [2, 1]], [[5, 0], [0, 1]]),
}
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
results = {}
for ax, (name, (A, B)) in zip(axes, panels.items(), strict=False):
    results[name] = br_picture(A, B, ax, name)
plt.tight_layout()
plt.show()

for name, res in results.items():
    print(name)
    for eq in res.equilibria:
        point = (str(eq["2"]["right"]), str(eq["1"]["Top"]))
        print("   ", show(eq), "  (prob(right), prob(Top)) =", point)

In the degenerate game (d), Gambit reports the two **extreme** equilibria, the endpoints
of the segment of equilibria on the right-hand edge, from $(1, 0)$ to $(1, 1/6)$. With
`cliques=True` it also reports which extreme equilibria can be combined (Chapter 6 calls
these maximal Nash subsets): here both endpoints form one set, so every convex
combination is also an equilibrium.

In [ ]:
[[show(eq) for eq in clique] for clique in results["(d) degenerate game"].cliques]

## 5.3 The best response condition for bimatrix games

A mixed strategy is a best response if and only if every pure strategy in its support is
a pure best response. Being indifferent between the strategies in the support is not
enough, as the chapter's $3\times 2$ example shows.

In [ ]:
g32 = bimatrix([[0, 4], [3, 0], [2, 2]], [[3, 0], [0, 3], [0, 3]],
               ["T", "M", "B"], ["l", "r"], title="Indifference is not enough")
y = [Fraction(4, 7), Fraction(3, 7)]
prof = g32.mixed_strategy_profile([[Fraction(1, 2), Fraction(1, 2), 0], y], rational=True)
print("Ay =", nice(prof.strategy_values)["1"])

Against $y = (4/7, 3/7)$ player 1 is indifferent between $T$ and $M$ (both $12/7$) but
the unique best response is $B$ (payoff 2). The upper envelope of player 1's payoff
lines, as a function of prob($r$), shows the pure best responses: $M$ up to $1/3$, $B$
between $1/3$ and $1/2$, $T$ beyond $1/2$.

In [ ]:
ax = plot_upper_envelope([(0, 4), (3, 0), (2, 2)], "prob(r)", "payoff to player 1",
                         ["T", "M", "B"], P1_COLOUR, marks=[Fraction(3, 7)])
ax.annotate("T and M cross at 3/7,\nbelow the envelope", (3 / 7, 12 / 7), (0.55, 0.6),
            arrowprops=dict(arrowstyle="->", color="0.4"), fontsize=8)
plt.show()

In [ ]:
def pure_best_responses(g, y):
    """The set of player 1's pure best responses to y (book: \\purebestresp(y))."""
    prof = g.mixed_strategy_profile([[1] + [0] * (len(g.get_strategies("1")) - 1), y],
                                    rational=True)
    vals = dict(prof.strategy_values["1"])
    best = max(vals.values())
    return {s for s, v in vals.items() if v == best}


pd.DataFrame(
    [{"prob(r)": str(p), "pure best responses": sorted(pure_best_responses(g32, [1 - p, p]))}
     for p in [Fraction(1, 4), Fraction(1, 3), Fraction(2, 5), Fraction(1, 2), Fraction(3, 4)]]
).set_index("prob(r)")

**Beyond the book.** The book does not solve this game; Gambit shows it has a unique
equilibrium, in which player 1 mixes $T$ and $B$:

In [ ]:
eq_table(gbt.nash.enummixed_solve(g32).equilibria)

## 5.4 Degenerate bimatrix games

A game is degenerate if some mixed strategy with support of size $k$ has more than $k$
pure best responses. In the chapter's degenerate $2\times 2$ game, both $T$ and $B$ are
best responses to the pure strategy $r$:

In [ ]:
degen = bimatrix([[0, 1], [2, 1]], [[5, 0], [0, 1]], ["T", "B"], ["l", "r"],
                 title="Degenerate 2x2 game")
print("payoffs to player 1 against r:",
      nice(degen.mixed_strategy_profile([[1, 0], [0, 1]], rational=True).strategy_values)["1"])
res = gbt.nash.enummixed_solve(degen, cliques=True)
print("extreme equilibria:", [show(eq) for eq in res.equilibria])
print("maximal Nash subsets:", [[show(eq) for eq in c] for c in res.cliques])

The full equilibrium set is $\{(x, r) : \mathrm{prob}(B) \in [5/6, 1]\}$, the segment
between the two extreme equilibria, as the book finds by hand.

## 5.5 Nash's Theorem for bimatrix games: Nash's map

Nash's map $f(x, y) = (x', y')$ shifts probability towards pure strategies that do
better than the current mixed strategy:
$\varphi_i = \max(0, (Ay)_i - x^\top A y)$, $x'_i = (x_i + \varphi_i) / (1 + \sum_k \varphi_k)$,
and similarly for player 2. Its fixed points are exactly the Nash equilibria. Gambit's
`strategy_values` and `payoffs` give $(Ay)_i$ and $x^\top A y$ directly, so the map works
for any number of players.

In [ ]:
def nash_map(profile):
    g = profile.game
    new = []
    for player in g.players:
        u = profile.payoffs[player]
        vals = profile.strategy_values[player]
        phi = {s: max(Fraction(0), frac(vals[s]) - frac(u)) for s in g.get_strategies(player)}
        total = 1 + sum(phi.values())
        new.append([(frac(profile[player][s]) + phi[s]) / total for s in g.get_strategies(player)])
    return g.mixed_strategy_profile(new, rational=True)


prof = mp.mixed_strategy_profile([[Fraction(3, 4), Fraction(1, 4)],
                                  [Fraction(1, 3), Fraction(2, 3)]], rational=True)
orbit = [prof]
for _ in range(3):
    orbit.append(nash_map(orbit[-1]))
for k, p in enumerate(orbit):
    print(f"iterate {k}:", show(p))

The first iterate is $x' = (1/2, 1/2)$, $y' = (1/4, 3/4)$, as in Example `ex:nashmap`, and
the next two are as stated there. Iterating further, the map circles around the
equilibrium $(1/2, 1/2)$ rather than converging to it. Brouwer's theorem only
guarantees that a fixed point exists; it does not say how to find one.

In [ ]:
prof = orbit[0].as_float()
pts = []
for _ in range(200):
    pts.append((prof["1"]["Heads"], prof["2"]["heads"]))
    g = prof.game
    new = []
    for player in g.players:
        u, vals = prof.payoffs[player], prof.strategy_values[player]
        phi = {s: max(0.0, vals[s] - u) for s in g.get_strategies(player)}
        new.append([(prof[player][s] + phi[s]) / (1 + sum(phi.values()))
                    for s in g.get_strategies(player)])
    prof = g.mixed_strategy_profile(new)
pts = np.array(pts)
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(pts[:, 0], pts[:, 1], ".-", color="0.5", lw=0.6, ms=3)
ax.plot(*pts[0], "o", color=P1_COLOUR, label="start")
ax.plot(0.5, 0.5, "o", color="black", label="equilibrium")
ax.set_xlabel("prob(Heads) for player 1")
ax.set_ylabel("prob(heads) for player 2")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.legend(fontsize=8)
plt.show()

## 5.6 Nash's Theorem for $n$-player games

The best response condition and Nash's Theorem hold for any number of players, and
Gambit handles $n$-player games too. With three or more players, however, the
equilibrium conditions are polynomial rather than linear, and equilibria can be
irrational (Remark `rem:irrationality` mentions Nash's three-player poker game).

**Beyond the book.** A three-player Stag Hunt with an irrational equilibrium. Each
hunter chooses Deer or Hare. Hare pays 1 for sure; Deer pays 2 if all three hunt deer,
and 0 otherwise.

In [ ]:
U = [np.empty((2, 2, 2), dtype=object) for _ in range(3)]
for s in np.ndindex(2, 2, 2):
    for i in range(3):
        U[i][s] = 1 if s[i] == 1 else (2 if s == (0, 0, 0) else 0)
stag3 = table_game(U, [["Deer", "Hare"]] * 3, title="Three-player Stag Hunt")
stag3

If the other two hunt deer with probabilities $q$ and $r$, Deer is worth $2qr$ and Hare
is worth 1, so a player mixes only if $qr = 1/2$. In a fully mixed equilibrium this holds
for all three pairs, so every player hunts deer with probability $1/\sqrt 2 \approx 0.7071$.

Gambit's `enumpoly_solve` (support enumeration with polynomial equations, Chapter 6)
finds the two pure equilibria and this irrational one, in floating point:

In [ ]:
res = gbt.nash.enumpoly_solve(stag3)
for eq in res.equilibria:
    print(show(eq, digits=10), "  max regret:", f"{eq.max_regret():.1e}")
print("1/sqrt(2) =", 1 / np.sqrt(2))